# 11 — Metrics & dashboard

CallMetricsAccumulator, MetricsStore, dashboard SSE, CSV/JSON export, pricing, basic auth.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises `CallMetricsAccumulator`, `MetricsStore`, and the CSV/JSON export helpers.


### CallMetricsAccumulator


In [ ]:
import time
from getpatter import CallMetricsAccumulator
with _setup.cell('metrics_accumulator', tier=1, env=env) as ok:
    if ok:
        acc = CallMetricsAccumulator(
            call_id='call-demo-001',
            provider_mode='openai_realtime',
            telephony_provider='twilio',
            stt_provider='deepgram',
            tts_provider='elevenlabs',
        )
        # Simulate a single turn.
        acc.start_turn()
        acc.record_stt_complete('What is the weather today?', audio_seconds=2.1)
        acc.record_llm_first_token()
        acc.record_llm_first_sentence()
        acc.record_tts_first_byte()
        tm = acc.record_turn_complete('The weather today is sunny and warm.')

        print(f'turn_index:      {tm.turn_index}')
        print(f'agent_text:      {tm.agent_text!r}')
        print(f'tts_characters:  {tm.tts_characters}')
        print(f'stt_audio_secs:  {tm.stt_audio_seconds:.1f}s')
        print(f'latency.total:   {tm.latency.total_ms:.0f}ms')
        assert tm.tts_characters == len('The weather today is sunny and warm.')
        assert tm.stt_audio_seconds == 2.1


### MetricsStore


In [ ]:
from getpatter import MetricsStore
with _setup.cell('metrics_store', tier=1, env=env) as ok:
    if ok:
        store = MetricsStore(max_calls=100)

        # Record two completed calls.
        for i in range(1, 3):
            store.record_call_start({
                'call_id': f'call-{i:03d}',
                'from': '+15555550100',
                'to': '+15555550200',
                'direction': 'inbound',
                'carrier': 'twilio',
                'provider': 'openai_realtime',
            })
            store.record_call_end({
                'call_id': f'call-{i:03d}',
                'duration': 30 * i,
                'cost_total': 0.05 * i,
            })

        calls = store.get_calls()
        agg   = store.get_aggregates()
        print(f'Total calls:   {agg["total_calls"]}')
        print(f'Avg duration:  {agg["avg_duration"]}s')
        print(f'Total cost:    ${agg["total_cost"]:.4f}')
        print(f'Active calls:  {agg["active_calls"]}')
        assert agg['total_calls'] == 2


### Export


In [ ]:
from getpatter import MetricsStore, calls_to_csv, calls_to_json
with _setup.cell('export', tier=1, env=env) as ok:
    if ok:
        store = MetricsStore()
        store.record_call_start({'call_id': 'c001', 'from': '+15555550100', 'to': '+15555550200', 'direction': 'inbound', 'carrier': 'twilio', 'provider': 'openai_realtime'})
        store.record_call_end({'call_id': 'c001', 'duration': 45, 'cost_total': 0.08})

        calls = store.get_calls()
        csv_output  = calls_to_csv(calls)
        json_output = calls_to_json(calls)

        csv_lines = csv_output.strip().splitlines()
        print(f'CSV header:  {csv_lines[0]}')
        print(f'CSV row:     {csv_lines[1]}')
        print(f'JSON output: {json_output[:80]}...')
        assert 'call_id' in csv_lines[0]
        assert 'c001' in csv_lines[1]
        assert 'c001' in json_output


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call and inspects the `MetricsStore` after it ends. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  metrics:  will inspect MetricsStore after call ends')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live call + metrics inspection *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, MetricsStore
with _setup.cell('live_metrics_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        store = MetricsStore(max_calls=50)
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a metrics demo agent. Greet and hang up.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        try:
            await p.call(env.target_number, agent=agent,
                         first_message='Hello from Patter metrics demo.',
                         ring_timeout=env.max_call_seconds)
            agg = store.get_aggregates()
            print(f'Calls in store:   {agg["total_calls"]}')
            print(f'Avg duration:     {agg["avg_duration"]}s')
            print(f'Total cost:       ${agg["total_cost"]:.4f}')
            print('✓ Metrics call completed')
        finally:
            _setup.hangup_leftover_calls(env)
